In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import bs4 as BeautifulSoup
from datetime import datetime
import os 
import requests
from copy import deepcopy

# region agent log
import json as _dbgjson, time as _dbgtime
_DBG_LOG_PATH = r"c:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\debug-05c009.log"
def _dbg(location, message, data=None, hypothesisId=None):
    try:
        with open(_DBG_LOG_PATH, "a", encoding="utf-8") as _f:
            _f.write(_dbgjson.dumps({"sessionId":"05c009","location":location,"message":message,"data":data or {},"hypothesisId":hypothesisId,"timestamp":int(_dbgtime.time()*1000)}, default=str) + "\n")
    except Exception:
        pass
# endregion


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'PT CMVM' ## change to current controller name
print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running PT CMVM Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict={

        regulatorName + ' 1': 'https://www.cmvm.pt/PInstitucional/Content?Input=EC2CC0691CC518A5DD27F600C912A72B95813803AA700D079B482F06F6A9D4D3',
        regulatorName + ' 2': 'https://www.cmvm.pt/PInstitucional/Content?Input=5DFE7211A0E7ECF9447CFDDEB7DF36130830865865A4241BD33893ED167B3991',
        regulatorName + ' 3': 'https://www.cmvm.pt/PInstitucional/Content?Input=DE69D31DE34B669FF11251BE9053B7BF8AF19CC5ECBEB434FC4E4A857452C478',
        regulatorName + ' 4': 'https://www.cmvm.pt/PInstitucional/Content?Input=E53C246B2452F00BEB3AEEF67C47C69CA0E3BB26DACB718CAD315564B7A421B4',
        regulatorName + ' 5': 'https://www.cmvm.pt/PInstitucional/Content?Input=185E6B62730853323BB4F2C2D727ACBA64584CC7E65E1DCDC5BDE00DA2BC7486',
        regulatorName + ' 6': 'https://www.cmvm.pt/PInstitucional/Content?Input=A569E0E41AC02102EA1881F785AF55733C331047139FDB06C96372769F55084F',
        regulatorName + ' 7': 'https://www.cmvm.pt/PInstitucional/Content?Input=6A7A54D1B9464E33BE3EBFA4FAA15FA2F45BE9F585803D7F3EFE1995C9FAD072'
        }



Typology={

       regulatorName + ' 1': 'Issuers',
       regulatorName + ' 2': 'Financial intermediaries registered with the CMVM',
       regulatorName + ' 3': 'Management Companies',
       regulatorName + ' 4': 'Investment funds',
       regulatorName + ' 5': 'Registered crowdfunding platform managers',
       regulatorName + ' 6': 'Financial intermediaries registered for providing investment advice',
       regulatorName + ' 7': 'Financial intermediaries that provide the services of investment research and financial analysis or other forms of general recommendation relating to transactions in financial instruments',

        }

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


In [ ]:



#------------------------------------------------ Begin_Main ----------------------------------------
# region agent log
_dbg("PT_CMVM_v2.ipynb:main_cell_entry", "main cell started", {"regdict_keys": list(regdict.keys())}, hypothesisId="sanity")
# endregion

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    driver.get(regdict[reg])
    time.sleep(10)
    if reg == regulatorName + ' 1':
        all_dfs = []
        wait = WebDriverWait(driver, 20)

        # 1. Wait for the pagination list to appear
        # We use a partial ID match ($=) because OutSystems IDs can change prefixes
        pagination_list_xpath = "//*[contains(@id, 'PaginationList')]"
        wait.until(EC.presence_of_element_located((By.XPATH, pagination_list_xpath)))

        # 2. Get all page buttons within that list
        # We also include the 'active' button which is usually just outside the 'list-group'
        tab_buttons =  driver.find_elements(By.XPATH, '//*[@data-block="Navigation.TabsHeaderItem"]')
        tab_buttons_num = len(tab_buttons)
        print(f"🎯 Detected {tab_buttons_num} tab buttons.")
        time.sleep(3)
        tables = driver.find_elements(By.XPATH, '//*[@id="b34-b6-tabscontent"]')[0].find_elements(By.CSS_SELECTOR, '[data-block="Navigation.TabsContentItem"]')
        for i in range(tab_buttons_num):
            print(f"🔘 Clicking tab button {i+1}...")
            
            time.sleep(3)
            header = driver.find_element(By.CSS_SELECTOR, "div.header-content.display-flex")
            header_h = header.size["height"]
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center', inline:'nearest'});", tab_buttons[i]
            )
            driver.execute_script("window.scrollBy(0, -arguments[0]);", header_h + 10)
            WebDriverWait(driver, 10).until(lambda d: tab_buttons[i].is_displayed() and tab_buttons[i].is_enabled())
            tab_buttons[i].click()
            time.sleep(3)
            tab_buttons[i].click()
            time.sleep(2)  # Wait for tab content to load
            pagination_list_xpath =  tables[i].find_element(By.CSS_SELECTOR, '[data-block="Navigation.Pagination"]').find_element(By.CSS_SELECTOR, 'div.pagination')
            total_pages = tables[i].find_element(By.CSS_SELECTOR, '[data-block="Navigation.Pagination"]').find_element(By.CSS_SELECTOR, 'div.pagination').get_attribute('data-totalpages')

            print(f"🎯 Detected {int(total_pages)+1} page.")
            # region agent log
            _dbg("PT_CMVM_v1.ipynb:tab_loop", "tab entered", {"tab_index": i, "total_pages_attr": total_pages, "loop_range": int(total_pages)}, hypothesisId="H3")
            # endregion

            for j in range(int(total_pages)):
                print(f"📄 Processing page {j+1}...")
                # region agent log
                _dbg("PT_CMVM_v1.ipynb:page_loop_enter", "page iter start", {"tab_index": i, "j": j, "total_pages": int(total_pages), "is_last_iter": (j == int(total_pages) - 1)}, hypothesisId="H2")
                # endregion
                            
                table = tables[i].find_element(By.TAG_NAME, "table")
                df = pd.read_html(table.get_attribute('outerHTML'))[0]
                all_dfs.append(df)
                print(f"✅ Page {j+1} scraped successfully.")

                pagination_list_xpath =  tables[i].find_element(By.CSS_SELECTOR, '[data-block="Navigation.Pagination"]').find_element(By.CSS_SELECTOR, 'div.pagination')
                # region agent log
                try:
                    _next_btn = pagination_list_xpath.find_element(By.CSS_SELECTOR,'button.pagination-button[aria-label*="go to next page"]')
                    _dbg("PT_CMVM_v1.ipynb:before_next_click", "next-button state pre-click", {
                        "tab_index": i, "j": j, "total_pages": int(total_pages),
                        "is_last_iter": (j == int(total_pages) - 1),
                        "disabled_attr": _next_btn.get_attribute("disabled"),
                        "aria_disabled": _next_btn.get_attribute("aria-disabled"),
                        "is_enabled_selenium": _next_btn.is_enabled(),
                        "outer_html_snippet": (_next_btn.get_attribute("outerHTML") or "")[:300],
                        "pagination_container_html_snippet": (pagination_list_xpath.get_attribute("outerHTML") or "")[:400],
                    }, hypothesisId="H1,H5")
                except Exception as _e:
                    _dbg("PT_CMVM_v1.ipynb:before_next_click_probe_failed", "probe failed", {"err": repr(_e)}, hypothesisId="H1")
                # endregion
                try:
                    pagination_list_xpath.find_element(By.CSS_SELECTOR,'button.pagination-button[aria-label*="go to next page"]').click()
                except Exception as _click_exc:
                    # region agent log
                    _dbg("PT_CMVM_v2.ipynb:click_exception", "click raised (guarded)", {"tab_index": i, "j": j, "total_pages": int(total_pages), "exc_type": type(_click_exc).__name__, "exc_msg": str(_click_exc)[:600]}, hypothesisId="H1,H5", )
                    # endregion
                    print(f"Error clicking next page: {_click_exc}")
                    continue

                # Wait for the table to refresh (AJAX)
                time.sleep(4) 
            table = tables[i].find_element(By.TAG_NAME, "table")
            df = pd.read_html(table.get_attribute('outerHTML'))[0]
            all_dfs.append(df)
            


            # Combine all data
            final_df = pd.concat(all_dfs, ignore_index=True)
            # final_df.to_csv("cmvm_all_pages.csv", index=False, encoding='utf-8-sig')
        print(f"🏆 Done! Total rows collected: {len(final_df)}")
        for name in final_df['Name']:
            sqldict['Name'].append(name)
            sqldict['ListName'].append(Typology[reg])
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0])
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict["Cntry"].append("PT")  
            sqldict = bourange_same_length_array(sqldict)
    elif reg == regulatorName + ' 2':
        all_dfs = []
        tab_lists = [
            "Top five trading platforms and order execution quality",
            "Qualifying holdings in financial intermediaries",
        ]
        while tab_lists:
            tab_text = tab_lists.pop(0)
            #print(tab_text)
            clicked = False
            tabs_list = driver.find_elements(By.CLASS_NAME, "accordion-sidemenu")
            for el in tabs_list:
                if tab_text in el.text:   # use "in" because el.text may include extra text/newlines
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                    el.click()
                    clicked = True
                    time.sleep(1)
                    break
            if not clicked:
                print("not found:", tab_text)
            time.sleep(3)  # Wait for tab content to load
            pagination_list_xpath =  driver.find_element(By.ID,"b4-MainContent").find_element(By.CSS_SELECTOR, '[data-block="Navigation.Pagination"]').find_element(By.CSS_SELECTOR, 'div.pagination')
            if pagination_list_xpath:
                total_pages = pagination_list_xpath.get_attribute('data-totalpages')
                print(f"🎯 Detected {int(total_pages)+1} page.")

                for j in range(int(total_pages)):
                    print(f"📄 Processing page {j+1}...")
                                
                    table = driver.find_element(By.TAG_NAME, 'table')
                    df = pd.read_html(table.get_attribute('outerHTML'))[0]
                    all_dfs.append(df)

                    print(f"✅ Page {j+1} scraped successfully.")

                    pagination_list_xpath =  driver.find_element(By.ID,"b4-MainContent").find_element(By.CSS_SELECTOR, '[data-block="Navigation.Pagination"]').find_element(By.CSS_SELECTOR, 'div.pagination')
                    try:
                        pagination_list_xpath.find_element(By.CSS_SELECTOR,'button.pagination-button[aria-label*="go to next page"]').click()
                    except Exception as e:
                        print(f"Error clicking next page: {e}")
                        continue

                    # Wait for the table to refresh (AJAX)
                    time.sleep(4) 
                table = driver.find_element(By.TAG_NAME, "table")
                df = pd.read_html(table.get_attribute('outerHTML'))[0]
                all_dfs.append(df)
            else:
                table = driver.find_element(By.TAG_NAME, 'table')
                df = pd.read_html(table.get_attribute('outerHTML'))[0]
                all_dfs.append(df)
                print(f"✅ Tab {tab_text} scraped successfully.")


            # Combine all data
            final_df = pd.concat(all_dfs, ignore_index=True)
            # final_df.to_csv("cmvm_all_pages.csv", index=False, encoding='utf-8-sig')
            print(f"🏆 Done! Total rows collected: {len(final_df)}")
        for name in final_df['Name']:
            sqldict['Name'].append(name)
            sqldict['ListName'].append(Typology[reg])
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0])
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict["Cntry"].append("PT")  
            sqldict = bourange_same_length_array(sqldict)

        POST_URL_LIST = 'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_IntermediariosFin_CW/Main/IntermediariosfinanceirosListWB/DataActionGet_Data'
        POST_URL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/IntermediarioFinanceirosDetail/DataActionGetIntermediarioFinanceiroByNum_ENT"
        REFERER  = "https://www.cmvm.pt/PInstitucional/Content?Input=B372DBB8D8C9718E94932DE8C7E598157C9601E33093C51036B192D9F0769616"
        list_payload = {
                "versionInfo": {
                    "moduleVersion": "wlyDNOTBlKBNwJYa303tDg",
                    "apiVersion": "FZtiBLy61eUFrZmiLRYS+g",
                },
                "viewName": "MainFlow.Content",
                "screenData": {
                    "variables": {
                        "StartIndex": 0,
                        "MaxRecord": 1000,
                        "Search_Name": '',
                        "_search_NameInDataFetchStatus": 1,
                        "Tipo": '',
                        "_tipoInDataFetchStatus": 1,
                        "ServicoRowNumber": 0,
                        "_servicoRowNumberInDataFetchStatus": 1,
                    }
                },
            }
        payload = {"versionInfo":{"moduleVersion":"wlyDNOTBlKBNwJYa303tDg","apiVersion":"7fiiD58ZQBsxmLUFa2In+g"},"viewName":"MainFlow.Content","screenData":{"variables":{"NUM_ENT":'',"_nUM_ENTInDataFetchStatus":1}}}

        s = requests.Session()
        s.cookies.update({
            "osVisit": "88283d6b-0f50-4eba-adbf-e0edd4d41562",
            "osVisitor": "1f90625d-94be-4d10-a8d2-b7a97a5454d6",
            "nr1Users": "lid%3dAnonymous%3btuu%3d0%3bexp%3d0%3brhs%3dXBC1ss1nOgYW1SmqUjSxLucVOAg%3d%3bhmc%3dEa1HzjBuUTncHx8CId%2bXGpoeEIQ%3d",
            "nr2Users": "crf%3dT6C%2b9iB49TLra4jEsMeSckDMNhQ%3d%3buid%3d0%3bunm%3d",
            "_pk_id.1.dc23": "d5e6c08ec4eafefb.1767022259.",
            "_pk_ses.1.dc23": "1",
        })

        headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "outsystems-request-token": "7701502406934197",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }

        r = s.post(POST_URL_LIST, json=list_payload, headers=headers)
        #print(r.status_code, r.text[:500])
        r.raise_for_status()
        data = r.json()
        nom_ent_list = data['data']['DataStructureList']['List']
        for li in nom_ent_list:
            name_ = li['NOM_ENT']
            #print(li['ENT_NUM_ENT'])
            payload['screenData']['variables']['NUM_ENT'] = li['ENT_NUM_ENT']
            r = s.post(POST_URL, json=payload, headers=headers)
            print(r.status_code, r.text[:500])
            r.raise_for_status()
            data = r.json()
            name_detail = data['data']['IntermediarioFinanceiro']['nom_ent']
            address_1 = data['data']['IntermediarioFinanceiro']['mor_ent']
            zip_code_ = data['data']['IntermediarioFinanceiro']['gr_cod_pos']
            city_ = data['data']['IntermediarioFinanceiro']['fr_cod_dsc']
            register_date = data['data']['IntermediarioFinanceiro']['inicio_act']
            tax_id = data['data']['IntermediarioFinanceiro']['num_ctb']
            topo = data['data']['IntermediarioFinanceiro']['tipo']

            sqldict['Name'].append(name_)
            sqldict['Address_1'].append(address_1)  
            sqldict['Zip'].append(zip_code_)
            sqldict['City'].append(city_)
            sqldict['RegulationDate'].append(register_date)
            sqldict['InternalID_1'].append(tax_id)
            sqldict['InternalID_1_type'].append('Tax identification number')
            sqldict['Typology'].append(topo)
            sqldict['ListName'].append(Typology[reg])
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0])
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict["Cntry"].append("PT")  
            sqldict = bourange_same_length_array(sqldict)
    elif reg == regulatorName + ' 3':

        api_dict = {
            "SociedadesGestoras": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetchSociedadesGestoras",
                                "token": "3706222453791299",
                                    "apiVersion": "PgGljBGtlHspvDFlgMm1Fw"
                                },
            "Empresas": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_EmpresasSeguros",
                        "token": "7701502406934197",
                        "apiVersion": "iAWZxkjpVvfaTAY9WAMoCg"},
            "STC": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_STC",
                    "token": "2616747537848431",
                    "apiVersion": "OJsVJq3Gu366Y7XC1R5eVA"},
            "EGFP": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_EGFP",
                    "token": "4027971824164562",
                    "apiVersion": "z9PkhiYn1+MuaB4nfPY0bg"},
            "SGFTC": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_SociedadesGestoras/DataActionFetch_SGFTC",
                    "token": "8040918438076384",
                    "apiVersion": "OFPYhNHcRG6a_g0p2sZigA"},
        }

        REFERER = "https://www.cmvm.pt/PInstitucional/Content?Input=DE69D31DE34B669FF11251BE9053B7BF8AF19CC5ECBEB434FC4E4A857452C478"

        payload_common = {"versionInfo":{"moduleVersion":"wlyDNOTBlKBNwJYa303tDg","apiVersion":"PgGljBGtlHspvDFlgMm1Fw"},
                            "viewName":"MainFlow.Content",
                            "screenData":{"variables":{"StartIndex_SociedadesGestoras":0,"MaxRecords":1000,"StartIndex_STC":0,"StartIndex_SGFTC":0,"StartIndex_EmpresasSeguros":0,"StartIndex_EGFP":0,"IsEmpty_SociedadesGestoras_1":'false',"IsEmpty_SGFTC_2":'false',"IsEmpty_STC_3":'false',"IsEmpty_ES_4":'false',"IsEmpty_EGFP_5":'false',
                                                        
                                                    "Entity_Name":"","_entity_NameInDataFetchStatus":1,"IsActive":'true',"_isActiveInDataFetchStatus":1,"Society_Type":"","_society_TypeInDataFetchStatus":1,"Dimension":"","_dimensionInDataFetchStatus":1,"Ambito":"","_ambitoInDataFetchStatus":1}}}


        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }

        # If each API needs a different StartIndex variable name, map it here (adjust if needed)
        start_index_key = {
            "SociedadesGestoras": "StartIndex_SociedadesGestoras",
            "Empresas": "StartIndex_EmpresasSeguros",
            "STC": "StartIndex_STC",
            "EGFP": "StartIndex_EGFP",
            "SGFTC": "StartIndex_SGFTC",
        }

        results = {}

        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}

            payload = deepcopy(payload_common)  # uses your existing payload_common
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]
            payload["screenData"]["variables"][start_index_key[api_name]] = 0  # optional

            r = s.post(api["url"], json=payload, headers=headers)
            print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()

        # results["SociedadesGestoras"], results["Empresas"], ...
        infors = []
        for result in results:
            print(result)
            data = results[result].get('data', {})
            # If 'List' is directly in data
            if 'List' in data:
                # print(data['List'])
                # print(type(data['List']))
                infors.append(data['List'])
                continue
            # Otherwise look for 'List' inside nested dict values
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    # print(v['List'])
                    # print(type(v['List']))  
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                # Fallback for debugging
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
            if result == "SociedadesGestoras" or result == "STC":
                if result == "SociedadesGestoras":
                    iterate_list =  infors[0]
                else:
                    iterate_list =  infors[2]
                for li in iterate_list:
                    update_code = li['NUM_ENT']
                    print(update_code)
                    # print(li['NOM_ENT'])
                    if result == "SociedadesGestoras":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_CapRiscoEmp_CW/UIFlow1/CapRiscoEmp_SociedadeDetail_Wb/DataActionGetSociedadeDetail"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "dm024DTlnLslZFDfoA_0cQ"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "Num_Ent": str(li["NUM_ENT"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    elif result == "STC":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/StcDetail/DataActionGetSct"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "52l_VyMgXpGRY3naCd3aWg"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_ENT": str(li["NUM_ENT"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                    # print(li["NUM_ENT"], r.status_code, r.text[:200])
                    r.raise_for_status()
                    detail_data = r.json()
                    if result == "SociedadesGestoras":
                        name_ = li['NOM_ENT']
                        address_1 = detail_data['data']['SociedadeDetails']['mor_ent']
                        zip_code_ = detail_data['data']['SociedadeDetails']['gr_cod_pos']
                        city_ = detail_data['data']['SociedadeDetails']['gr_cod_dsc']
                        register_date = detail_data['data']['SociedadeDetails']['data_reg']
                        tax_id = detail_data['data']['SociedadeDetails']['num_ctb']
                        tipo = detail_data['data']['SociedadeDetails']['tipo']
                        print(name_, address_1, zip_code_, city_, register_date, tax_id, tipo)
                    elif result == "STC":
                        name_ = li['NOM_ENT']
                        address_1 = detail_data['data']['Sct']['mor_stc']
                        zip_code_ = detail_data['data']['Sct']['cod_pst']
                        city_ = detail_data['data']['Sct']['gr_cod_dsc_abr']
                        register_date = detail_data['data']['Sct']['dat_reg']
                        register_number = detail_data['data']['Sct']['num_reg']
                        tax_id = detail_data['data']['Sct']['num_ctb']
                        tipo = detail_data['data']['Sct']['tipo']
                        print(name_, address_1, zip_code_, city_, register_date, tax_id, tipo)

                    sqldict['Name'].append(name_)
                    sqldict['Address_1'].append(address_1)  
                    sqldict['Zip'].append(zip_code_)
                    sqldict['City'].append(city_)
                    sqldict['RegulationDate'].append(register_date)
                    sqldict['InternalID_1'].append(tax_id)
                    sqldict['InternalID_1_type'].append('Tax identification number')
                    sqldict['Typology'].append(tipo)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
            elif result == "Empresas" or result == "EGFP" :
                if result == "Empresas":
                    iterate_list =  infors[1]
                elif result == "EGFP":
                    iterate_list =  infors[3]

                    
                for li in iterate_list:
                    update_code = li['Value']
                    #print(update_code)
                    if result == "Empresas":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/EmpresaSeguroDetail/DataActionGetData"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "eGyCcqSS6uhvu9nqqGAMUQ"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_ENT": str(li["Value"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    elif result == "EGFP":
                        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/EmpresaSeguroDetail/DataActionGetData"
                        payload_detail = {
                        "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "eGyCcqSS6uhvu9nqqGAMUQ"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_ENT": str(li["Value"]),
                                "_num_EntInDataFetchStatus": 1,
                                "Tipo": "Venture Capital Company",
                                "_tipoInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                    # print(li["NUM_ENT"], r.status_code, r.text[:200])
                    r.raise_for_status()
                    detail_data = r.json()
                    # if result == "Empresas":
                    name_ = detail_data['data']['EmpresaSeguro']['Nom_ent']
                    address_1 = detail_data['data']['EmpresaSeguro']['mor_ent']
                    zip_code_ = detail_data['data']['EmpresaSeguro']['Cod_pst']
                    city_ = detail_data['data']['EmpresaSeguro']['gr_cod_dsc']
                    email_ = detail_data['data']['EmpresaSeguro']['email']
                    website_ = detail_data['data']['EmpresaSeguro']['site_ent']

                    print(name_,address_1, zip_code_, city_, email_, website_)
                    sqldict['Name'].append(name_)
                    sqldict['Address_1'].append(address_1)  
                    sqldict['Zip'].append(zip_code_)
                    sqldict['City'].append(city_)
                    sqldict['Email'].append(email_)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCode'].append('3')
                    sqldict['ListCode'].append('3')
                    sqldict["Cntry"].append("PT")  
                    sqldict = bourange_same_length_array(sqldict)
            elif result == "SGFTC":
                iterate_list =  infors[4]
                for li in iterate_list:
                    name_ = li['Text']
                    print(name_)
                    sqldict['Name'].append(name_)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')

                    sqldict['RegCode'].append('3')
                    sqldict['ListCode'].append('3')
                    sqldict["Cntry"].append("PT")  
                    sqldict = bourange_same_length_array(sqldict)
    elif reg == regulatorName + ' 4':      
        api_dict = {
            "Pensoes": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosPensoes",
                                "token": "7011207507672761",
                                "apiVersion": "_Pgt+oS9kB0YCiQR5L8OBA"
                                },
            "RecuperaCreditos": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosRecuperaCreditos",
                        "token": "6119502279184928",
                        "apiVersion": "o_5_n+NaxoawcbIxlzMN7Q"},
            "Mobiliario": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosInvestimentoMobiliario",
                    "token": "6832074922633300",
                    "apiVersion": "2Kyh7s0bYwCUR_MKDQ02nQ"},
            "Investimento": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchSociedadesInvestimento",
                    "token": "1048115306215631",
                    "apiVersion": "TUTaqt2hCTUkF8T98ecIdQ"},
            "CapitalRisco": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosCapitalRisco",
                    "token": "8606747129710045",
                    "apiVersion": "aMaXSzQtee_WdMh_bxtgew"},
            "FundosInvestimentoIMobiliario": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundosInvestimentoIMobiliario",
                    "token": "8665773788672252",
                    "apiVersion": "lHjTvAr3axK4WTTLmOMdwQ"},
            "FundosTitularizacaoCredito": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchTitularizacaoCredito",
                    "token": "2730594537642582",
                    "apiVersion": "A7L1T3Y6Y4sxiOT1Qts+1Q"},
            "SocialeAlternativo": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/RGA/Fun_GestAtivos_FundosInvestimento/DataActionFetchFundoEmpreendedorismoSocialeAlternativo",
                "token": "2526903115018427",
                "apiVersion": "xOdD+tVclDPcmX+gYQcDPQ"},
        }

        REFERER = "https://www.cmvm.pt/PInstitucional/Content?Input=E53C246B2452F00BEB3AEEF67C47C69CA0E3BB26DACB718CAD315564B7A421B4"

        payload_common = {
        "versionInfo": {"moduleVersion":"wlyDNOTBlKBNwJYa303tDg","apiVersion":"xOdD+tVclDPcmX+gYQcDPQ"},
        "viewName":"MainFlow.Content",
        "screenData":{"variables":{
            "MaxRecords":1000,
            "StartIndex_FII":0,
            "StartIndex_FIM":0,
            "StartIndex_FCR":0,
            "StartIndex_SIC":0,
            "StartIndex_FTC":0,
            "StartIndex_FRC":0,
            "StartIndex_FP":0,
            "StartIndex_FC":0,
            "StartIndex_FES":0,
            "StartIndex_SES":0,
            "Input_IsActive": True,
            "_input_IsActiveInDataFetchStatus": 1,
            "Input_Num_Fundo": "",
            "_input_Num_FundoInDataFetchStatus": 1,
            "Input_Nom_SubFundo": "",
            "_input_Nom_SubFundoInDataFetchStatus": 1,
            "Input_Num_EntGestora": "",
            "_input_Num_EntGestoraInDataFetchStatus": 1
        }}
        }

        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }

        results = {}

        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}

            payload = deepcopy(payload_common)  # uses your existing payload_common
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]  # optional
            s = requests.Session()
            s.cookies.update({
                "osVisit": "88283d6b-0f50-4eba-adbf-e0edd4d41562",
                "osVisitor": "1f90625d-94be-4d10-a8d2-b7a97a5454d6",
                "nr1Users": "lid%3dAnonymous%3btuu%3d0%3bexp%3d0%3brhs%3dXBC1ss1nOgYW1SmqUjSxLucVOAg%3d%3bhmc%3dEa1HzjBuUTncHx8CId%2bXGpoeEIQ%3d",
                "nr2Users": "crf%3dT6C%2b9iB49TLra4jEsMeSckDMNhQ%3d%3buid%3d0%3bunm%3d",
                "_pk_id.1.dc23": "d5e6c08ec4eafefb.1767022259.",
                "_pk_ses.1.dc23": "1",
            })

            headers = {
                "Accept": "application/json",
                "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
                "Content-Type": "application/json; charset=UTF-8",
                "Origin": "https://www.cmvm.pt",
                "Referer": REFERER,
                "outsystems-locale": "en-US",
                "outsystems-request-token": "7701502406934197",
                "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
                "x-ps-ext": "2024",
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
            }
            r = s.post(api["url"], json=payload, headers=headers)
            print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()

        # results["SociedadesGestoras"], results["Empresas"], ...
        infors = []
        for result in results:
            print(result)
            data = results[result].get('data', {})
            # If 'List' is directly in data
            if 'List' in data:
                # print(data['List'])
                # print(type(data['List']))
                infors.append(data['List'])
                continue
            # Otherwise look for 'List' inside nested dict values
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    # print(v['List'])
                    # print(type(v['List']))  
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                # Fallback for debugging
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")


        for k, v in results.items():
        #print(v['data'])
        #print(k)
        #Pension funds - get
            if k == "Pensoes":
                iterate_list =  infors[0]
                POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/Fundo_Pensao_Detail/DataActionGetFundo"
                for li in iterate_list:
                    update_code = li['num_prd']
                    print(update_code)
                    payload_detail = {
                        "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "cnDvjEOyf_gWI8s63cVoOA"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_FUN": str(li["num_prd"]),
                                "_nUM_FUNInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                    # print(li["NUM_ENT"], r.status_code, r.text[:200])
                    r.raise_for_status()
                    detail_data = r.json()
                    # if result == "Empresas":
                    name_ = detail_data['data']['Fundo']['nom_prd']
                    management_company_ = detail_data['data']['Fundo']['nom_ent']
                    tipo_ = detail_data['data']['Fundo']['desc_fun']
                    #print(name_, management_company_,  tipo_)
                    sqldict['Name'].append(name_)
                    sqldict['Name - Mother Company'].append(management_company_)
                    sqldict['Typology'].append(tipo_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)

            #Securities investment funds & Real estate investment funds & Recovery loan funds - all get
            elif k == "Mobiliario" or k == 'FundosInvestimentoIMobiliario' or k == "RecuperaCreditos" or k == "Investimento" or k == "CapitalRisco" or k == "SocialeAlternativo":
                if k == "Mobiliario":
                    iterate_list =  infors[2]
                elif k == "RecuperaCreditos":
                    iterate_list =  infors[1]
                elif k == "Investimento":
                    iterate_list =  infors[3]
                elif k == "CapitalRisco":
                    iterate_list =  infors[4]
                elif k == 'FundosInvestimentoIMobiliario':
                    iterate_list =  infors[5]
                elif k == "SocialeAlternativo":
                    iterate_list =  infors[7]
                POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_CapRiscoEmp_CW/UIFlow1/CapRiscoEmp_FundoDetail_Wb/DataActionGetFundoDetail"
                for li in iterate_list:
                    update_code = li['NUM_FUN']
                    #print(update_code)
                    payload_detail = {
                        "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "kd9XGHm9TtC4HItj1nEF8g"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "Fun_Num": str(li["NUM_FUN"]),
                                "_fun_NumInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                    # print(li["NUM_ENT"], r.status_code, r.text[:200])
                    r.raise_for_status()
                    detail_data = r.json()
                    # if result == "Empresas":
                    name_ = detail_data['data']['Out_FundoDetail']['descom_fun']
                    management_company_ = detail_data['data']['Out_FundoDetail']['ent_gestora']
                    isin_code_ = detail_data['data']['Out_FundoDetail']['cod_isi']
                    register_date_ = detail_data['data']['Out_FundoDetail']['data_inicio']
                    fund_code = detail_data['data']['Out_FundoDetail']['num_fun']
                    tipo_ = detail_data['data']['Out_FundoDetail']['tipo']
                    sqldict['Name'].append(name_)
                    sqldict['Name - Mother Company'].append(management_company_)
                    sqldict['InternalID_1'].append(isin_code_)
                    sqldict['InternalID_1_type'].append('ISIN code')
                    sqldict['RegulationDate'].append(register_date_)
                    sqldict['InternalID_2'].append(fund_code)
                    sqldict['InternalID_2_type'].append('Fund code')
                    sqldict['Typology'].append(tipo_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
                    #print(name_, management_company_, isin_code_, register_date_, fund_code, tipo_)

            #Securitisation funds - get
            elif k == "FundosTitularizacaoCredito":
                iterate_list =  infors[6]
                POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_Fun_GestAtivos_CW/Detail/FundoDetail/DataActionGetFundo"
                for li in iterate_list:
                    update_code = li['Id']
                    #print(update_code)
                    payload_detail = {
                        "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "9OHQIJUB2i1HouF02khGxg"},
                        "viewName": "MainFlow.Content",
                        "screenData": {
                            "variables": {
                                "NUM_FUN": str(li["Id"]),
                                "_nUM_FUNInDataFetchStatus": 1,
                            }
                        },
                    }
                    r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                    # print(li["NUM_ENT"], r.status_code, r.text[:200])
                    r.raise_for_status()
                    detail_data = r.json()
                    # if result == "Empresas":
                    name_ = detail_data['data']['Fundo']['nome']
                    management_company_ = detail_data['data']['Fundo']['gestora']
                    isin_code_ = detail_data['data']['Fundo']['cod_isi']
                    register_date_ = detail_data['data']['Fundo']['data_inicio']
                    tipo_ = detail_data['data']['Fundo']['tipo']
                    #print(name_, management_company_, isin_code_, register_date_, tipo_)
                    sqldict['Name'].append(name_)
                    sqldict['Name - Mother Company'].append(management_company_)
                    sqldict['InternalID_1'].append(isin_code_)
                    sqldict['InternalID_1_type'].append('ISIN code')
                    sqldict['RegulationDate'].append(register_date_)
                    sqldict['Typology'].append(tipo_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])       
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
    elif reg == regulatorName + ' 5':
        api_dict = {
            "Crowdfunding": {"url": "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_FinColaborativo_CW/UIFlow1/RegisteredEntityCrowdfunding/DataActionGetRegisteredEntityCrowdfunding",
                                "token": "1627131176195664",
                                    "apiVersion": "E+dt2ieyrhwZ3OGir57dxQ"
                                }}
        payload_common = {"versionInfo":{"moduleVersion":"wlyDNOTBlKBNwJYa303tDg","apiVersion":"PgGljBGtlHspvDFlgMm1Fw"},
                            "viewName":"MainFlow.Content",
                            "screenData":{"variables":{"StartIndex":0,"MaxRecords":1000,}}}
        REFERER = "https://www.cmvm.pt/PInstitucional/Content?Input=185E6B62730853323BB4F2C2D727ACBA64584CC7E65E1DCDC5BDE00DA2BC7486"
        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }
        results = {}
        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}

            payload = deepcopy(payload_common)  # uses your existing payload_common
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]

            r = s.post(api["url"], json=payload, headers=headers)
            print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()

        # results["SociedadesGestoras"], results["Empresas"], ...
        infors = []
        for result in results:
            print(result)
            data = results[result].get('data', {})
            # If 'List' is directly in data
            if 'List' in data:
                # print(data['List'])
                # print(type(data['List']))
                infors.append(data['List'])
                continue
            # Otherwise look for 'List' inside nested dict values
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    # print(v['List'])
                    # print(type(v['List']))  
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                # Fallback for debugging
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
        iterate_list =  infors[0]
        POST_URL_DETAIL = "https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_FinColaborativo_CW/UIFlow1/RegisteredEntityCrowdfunding_Detail/DataActionGetRegisteredCrowdfundingPlatform"
        for li in iterate_list:
                update_code = li['num_ent']
                print(update_code)
                payload_detail = {
                    "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "DjQJaplTYz_oV+8CsmnMVA"},
                    "viewName": "MainFlow.Content",
                    "screenData": {
                        "variables": {
                            "NUM_ENT": str(li["num_ent"]),
                            "_nUM_ENTInDataFetchStatus": 1,
                        }
                    },
                }
                r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                # print(li["NUM_ENT"], r.status_code, r.text[:200])
                r.raise_for_status()
                detail_data = r.json()
                name_ = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['NOM_ENT']
                tip_fin = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['TIP_FIN']
                mor_ent = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['MOR_ENT']
                cod_pst = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['COD_PST']
                gr_cod_dsc = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['GR_COD_DSC']
                dat_reg = detail_data['data']['RegisteredCrowdfundingPlatform_Detail']['DAT_REG']
                # print(name_, tip_fin, mor_ent, cod_pst, gr_cod_dsc, dat_reg)
                sqldict['Name'].append(name_)
                sqldict['Typology'].append(tip_fin)
                sqldict['Address_1'].append(mor_ent)
                sqldict['Zip'].append(cod_pst)
                sqldict['City'].append(gr_cod_dsc)
                sqldict['RegulationDate'].append(dat_reg)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
    elif reg == regulatorName + ' 6':
        api_dict = {
            "Financial intermediaries registered for providing investment advices": {"url": 'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/FinancialIntermediaryList_CW/DataActionFetchFinancialIntermediaryList',
                                    "token": "2280863588950865",
                                    "apiVersion": "nEjGGjihelof6xGGKHyoaw"
                                }}
        payload_common = {"versionInfo":{"moduleVersion":"wlyDNOTBlKBNwJYa303tDg","apiVersion":"nEjGGjihelof6xGGKHyoaw"},"viewName":"MainFlow.Content","screenData":{"variables":{"StartIndex":0,"MaxRecord":1000,"ServNum":5,"_servNumInDataFetchStatus":1,"Search_EntityName":"","_search_EntityNameInDataFetchStatus":1,"Search_Type":"","_search_TypeInDataFetchStatus":1}}}
        REFERER = "https://www.cmvm.pt/PInstitucional/Content?Input=D40F6BF51303623A87BD0F0B9EC81578C0F3FEA37E17CAE285909811F7E2C866"
        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }
        results = {}
        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}

            payload = deepcopy(payload_common)  # uses your existing payload_common
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]

            r = s.post(api["url"], json=payload, headers=headers)
            #print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()
        infors = []
        for result in results:
            #print(result)
            data = results[result].get('data', {})
            # If 'List' is directly in data
            if 'List' in data:
                # print(data['List'])
                # print(type(data['List']))
                infors.append(data['List'])
                continue
            # Otherwise look for 'List' inside nested dict values
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    # print(v['List'])
                    # print(type(v['List']))  
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                # Fallback for debugging
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
        iterate_list =  infors[0]
        POST_URL_DETAIL =  'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/IntermediarioFinanceirosDetail/DataActionGetIntermediarioFinanceiroByNum_ENT'
        for li in iterate_list:
                update_code = li['NUM_ENT']
                #print(update_code)
                payload_detail = {
                    "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "7fiiD58ZQBsxmLUFa2In+g"},
                    "viewName": "MainFlow.Content",
                    "screenData": {
                        "variables": {
                            "NUM_ENT": str(li["NUM_ENT"]),
                            "_nUM_ENTInDataFetchStatus": 1,
                        }
                    },
                }
                r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                # print(li["NUM_ENT"], r.status_code, r.text[:200])
                r.raise_for_status()
                detail_data = r.json()
                name_ = detail_data['data']['IntermediarioFinanceiro']['nom_ent']
                address_1 = detail_data['data']['IntermediarioFinanceiro']['mor_ent']
                zip_code_ = detail_data['data']['IntermediarioFinanceiro']['gr_cod_pos']
                city_ = detail_data['data']['IntermediarioFinanceiro']['fr_cod_dsc']
                tipo = detail_data['data']['IntermediarioFinanceiro']['tipo']
                email_ = detail_data['data']['IntermediarioFinanceiro']['email']
                tax_id = detail_data['data']['IntermediarioFinanceiro']['num_ctb']
                inner_num =  detail_data['data']['IntermediarioFinanceiro']['num_reg']

                sqldict['Name'].append(name_)
                sqldict['Typology'].append(tipo)
                sqldict['Address_1'].append(address_1)
                sqldict['Zip'].append(zip_code_)
                sqldict['City'].append(city_)
                sqldict['Email'].append(email_)
                sqldict['InternalID_1'].append(tax_id)
                sqldict['InternalID_1_type'].append('Tax identification number')
                sqldict['InternalID_2'].append(inner_num)
                sqldict['InternalID_2_type'].append('Registration number with the CMVM')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)   
    elif reg == regulatorName + ' 7':
        api_dict = {
            "Financial intermediaries registered for providing investment advices": {"url": 'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_AnalFinanceiros_CW/Main/IntermediariosFinanceirosList_Wb/DataActionGetIntermediariosFinanceiros',
                                    "token": "6322880696288143",
                                    "apiVersion": "YLLqP7MQZUzxA7E74_xhMA"
                                }}
        payload_common ={"versionInfo":{"moduleVersion":"wlyDNOTBlKBNwJYa303tDg","apiVersion":"YLLqP7MQZUzxA7E74_xhMA"},"viewName":"MainFlow.Content","screenData":{"variables":{"StartIndex":0,"MaxRecords":1000,"Search_EntityName":"","_search_EntityNameInDataFetchStatus":1,"Search_Type":"","_search_TypeInDataFetchStatus":1}}}
        REFERER = "https://www.cmvm.pt/PInstitucional/Content?Input=6A7A54D1B9464E33BE3EBFA4FAA15FA2F45BE9F585803D7F3EFE1995C9FAD072"
        base_headers = {
            "Accept": "application/json",
            "Accept-Language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
            "Content-Type": "application/json; charset=UTF-8",
            "Origin": "https://www.cmvm.pt",
            "Referer": REFERER,
            "outsystems-locale": "en-US",
            "x-csrftoken": "T6C+9iB49TLra4jEsMeSckDMNhQ=",
            "x-ps-ext": "2024",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        }
        results = {}
        for api_name, api in api_dict.items():
            headers = {**base_headers, "outsystems-request-token": api["token"]}

            payload = deepcopy(payload_common)  # uses your existing payload_common
            payload["versionInfo"]["apiVersion"] = api["apiVersion"]

            r = s.post(api["url"], json=payload, headers=headers)
            #print(api_name, r.status_code, r.text[:200])
            r.raise_for_status()
            results[api_name] = r.json()
        infors = []
        for result in results:
            print(result)
            data = results[result].get('data', {})
            # If 'List' is directly in data
            if 'List' in data:
                # print(data['List'])
                # print(type(data['List']))
                infors.append(data['List'])
                continue
            # Otherwise look for 'List' inside nested dict values
            found = False
            for v in data.values():
                if isinstance(v, dict) and 'List' in v:
                    # print(v['List'])
                    # print(type(v['List']))  
                    infors.append(v['List'])
                    found = True
                    break
            if not found:
                # Fallback for debugging
                print(f"No 'List' found in results[{result}]. Keys: {list(data.keys())}")
        iterate_list =  infors[0]
        POST_URL_DETAIL =  'https://www.cmvm.pt/PInstitucional/screenservices/CMVM_SDI_ConsultoresAutonomos_CW/MainFlowTest/IntermediarioFinanceirosDetail/DataActionGetIntermediarioFinanceiroByNum_ENT'
        for li in iterate_list:
                update_code = li['Entity_Number']
                #print(update_code)
                payload_detail = {
                    "versionInfo": {"moduleVersion": "wlyDNOTBlKBNwJYa303tDg", "apiVersion": "7fiiD58ZQBsxmLUFa2In+g"},
                    "viewName": "MainFlow.Content",
                    "screenData": {
                        "variables": {
                            "NUM_ENT": str(li["Entity_Number"]),
                            "_nUM_ENTInDataFetchStatus": 1,
                        }
                    },
                }
                r = s.post(POST_URL_DETAIL, json=payload_detail, headers=headers)
                # print(li["NUM_ENT"], r.status_code, r.text[:200])
                r.raise_for_status()
                detail_data = r.json()
                name_ = detail_data['data']['IntermediarioFinanceiro']['nom_ent']
                address_1 = detail_data['data']['IntermediarioFinanceiro']['mor_ent']
                zip_code_ = detail_data['data']['IntermediarioFinanceiro']['gr_cod_pos']
                city_ = detail_data['data']['IntermediarioFinanceiro']['fr_cod_dsc']
                tipo = detail_data['data']['IntermediarioFinanceiro']['tipo']
                email_ = detail_data['data']['IntermediarioFinanceiro']['email']
                tax_id = detail_data['data']['IntermediarioFinanceiro']['num_ctb']
                
                sqldict['Name'].append(name_)
                sqldict['Typology'].append(tipo)
                sqldict['Address_1'].append(address_1)
                sqldict['Zip'].append(zip_code_)
                sqldict['City'].append(city_)
                sqldict['Email'].append(email_)
                sqldict['InternalID_1'].append(tax_id)
                sqldict['InternalID_1_type'].append('Tax identification number')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)   

[INFO] : Working 1/7 _(PT CMVM 1)_ 
🎯 Detected 2 tab buttons.
🔘 Clicking tab button 1...
🎯 Detected 3 page.
📄 Processing page 1...
✅ Page 1 scraped successfully.


C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28988\3523302833.py:47: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table.get_attribute('outerHTML'))[0]


📄 Processing page 2...
✅ Page 2 scraped successfully.


C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28988\3523302833.py:47: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table.get_attribute('outerHTML'))[0]
C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28988\3523302833.py:57: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table.get_attribute('outerHTML'))[0]


🔘 Clicking tab button 2...
🎯 Detected 15 page.
📄 Processing page 1...
✅ Page 1 scraped successfully.


...

📄 Processing page 2...
✅ Page 2 scraped successfully.


C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28988\3523302833.py:106: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table.get_attribute('outerHTML'))[0]
C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28988\3523302833.py:116: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table.get_attribute('outerHTML'))[0]


🏆 Done! Total rows collected: 87
not found: Qualifying holdings in financial intermediaries
🎯 Detected 3 page.
📄 Processing page 1...
✅ Page 1 scraped successfully.


C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28988\3523302833.py:106: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table.get_attribute('outerHTML'))[0]


ElementClickInterceptedException: Message: element click intercepted: Element <button data-button="" class="pagination-button" type="button" disabled="" aria-label="go to next page">...</button> is not clickable at point (410, 301). Other element would receive the click: <div data-container="" class="pagination-container OSInline" role="navigation" aria-label="Pagination" id="b120-b10-b1-PaginationContainer">...</div>
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6aa3e7de5+14895]
	chromedriver!GetHandleVerifier [0x7ff6aa3e7e50+14900]
	chromedriver!(No symbol) [0x7ff6aa14d5ad]
	chromedriver!(No symbol) [0x7ff6aa1af3b5]
	chromedriver!(No symbol) [0x7ff6aa1acfca]
	chromedriver!(No symbol) [0x7ff6aa1aa307]
	chromedriver!(No symbol) [0x7ff6aa1a91b7]
	chromedriver!(No symbol) [0x7ff6aa19bce6]
	chro

In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
sqldict = bourange_same_length_array(sqldict)   
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)

# driver.quit()


In [ ]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 208 values.
Key 'priority' has 208 values.
Key 'ListLabel' has 208 values.
Key 'Typology' has 208 values.
Key 'EntryType' has 208 values.
Key 'Name' has 208 values.
Key 'InternalID_1' has 208 values.
Key 'InternalID_1_type' has 208 values.
Key 'InternalID_2' has 208 values.
Key 'InternalID_2_type' has 208 values.
Key 'InternalID_3' has 208 values.
Key 'InternalID_3_type' has 208 values.
Key 'CoType' has 208 values.
Key 'License_Type' has 208 values.
Key 'Address_1' has 208 values.
Key 'Address_2' has 208 values.
Key 'City' has 208 values.
Key 'Zip' has 208 values.
Key 'Cntry' has 208 values.
Key 'Phone' has 208 values.
Key 'Fax' has 208 values.
Key 'Website' has 208 values.
Key 'Email' has 208 values.
Key 'RegulationType' has 208 values.
Key 'RegulationTypeCode' has 208 values.
Key 'RegulationDate' has 208 values.
Key 'CancellationDate' has 208 values.
Key 'RegCtry' has 208 values.
Key 'RegCode' has 208 values.
Key 'ListCode' has 208 values.
Key 'ListLanguage' has 208 v